### Restart and Run All Cells

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
year = 2025
quarter = 4
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:02 10:42:11


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',    
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%'
              }

In [4]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2025 AND quarter <= 4) 
OR (year = 2025-1 AND quarter >= 4+1)
ORDER BY year DESC, quarter DESC


In [5]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8100,10,"10,827,771",4
2,ADVANC,8100,10,"47,885,902",4
4,AH,8100,10,"731,428",4
6,AIMIRT,8100,10,"684,130",4
11,AOT,8100,10,"18,125,205",4


In [6]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s-1 AND quarter <= %s) 
OR (year = %s-2 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2025-1 AND quarter <= 4) 
OR (year = 2025-2 AND quarter >= 4+1)
ORDER BY year DESC, quarter DESC



In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8096,10,"5,279,119",4
1,ACE,8096,10,"838,717",4
2,ADVANC,8096,10,"35,075,357",4
3,AEONTS,8096,10,"2,860,344",4
4,AH,8096,10,"746,961",4


In [8]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
2,AH,2025,Q4,"731,428","746,961","-15,533",-2.08%
3,AIMIRT,2025,Q4,"684,130","948,696","-264,566",-27.89%
4,AOT,2025,Q4,"18,125,205","19,182,395","-1,057,190",-5.51%


In [9]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM yr_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 140


In [10]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
rcds = df_ins.values.tolist()
len(rcds)

141

In [11]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO yr_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [13]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
2,AH,2025,Q4,"731,428","746,961","-15,533",-2.08%
3,AIMIRT,2025,Q4,"684,130","948,696","-264,566",-27.89%
4,AOT,2025,Q4,"18,125,205","19,182,395","-1,057,190",-5.51%


In [14]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
2,AH,2025,Q4,"731,428","746,961","-15,533",-2.08%
3,AIMIRT,2025,Q4,"684,130","948,696","-264,566",-27.89%
4,AOT,2025,Q4,"18,125,205","19,182,395","-1,057,190",-5.51%


In [15]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
7,ASK,2025,Q4,"531,545","331,797","199,748",60.20%
12,BAM,2025,Q4,"1,812,270","1,601,642","210,628",13.15%
16,BCP,2025,Q4,"2,879,701","2,184,088","695,613",31.85%


In [16]:
final_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[final_criteria, cols].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
39,DIF,2025,Q4,"13,855,643","656,288","13,199,355",2011.21%
48,GULF,2025,Q4,"101,380,618","18,170,334","83,210,284",457.95%
28,CK,2025,Q4,"3,328,223","1,445,903","1,882,320",130.18%
79,OSP,2025,Q4,"3,666,567","1,638,105","2,028,462",123.83%
98,SCC,2025,Q4,"14,075,020","6,341,638","7,733,382",121.95%


In [17]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
12,BAM,2025,Q4,"1,812,270","1,601,642","210,628",13.15%
16,BCP,2025,Q4,"2,879,701","2,184,088","695,613",31.85%
24,BLA,2025,Q4,"6,968,375","4,317,070","2,651,305",61.41%


In [18]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
12,BAM,2025,Q4,"1,812,270","1,601,642","210,628",13.15%
16,BCP,2025,Q4,"2,879,701","2,184,088","695,613",31.85%
24,BLA,2025,Q4,"6,968,375","4,317,070","2,651,305",61.41%


In [19]:
conlt.commit()
conlt.close()

In [20]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:02 10:42:12
